[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/extract_pitch_frames.ipynb)

# Pitch-frame extraction (Phase 3.5b)
Samples frames from a Trace game for pitch-keypoint labeling in Roboflow.
Stratified to over-sample hard frames (end-zone pans, shadow boundaries).
Do NOT run on the bake-off clip or held-out games — training games only.


In [ ]:
from pathlib import Path
import cv2

VIDEO = Path("/content/game.mp4")   # a TRAINING game, not the bake-off clip
OUT = Path("/content/frames"); OUT.mkdir(exist_ok=True)
STRIDE = 30                         # ~1 fps base sampling at 30fps
MAX_FRAMES = 400


In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
saved = 0
for i in range(0, n, STRIDE):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, frame = cap.read()
    if not ok:
        continue
    # crude pan-position proxy: column of brightest green mass (field center).
    # Frames where the field center is near an edge are end-zone pans -> keep all;
    # otherwise keep every other one to avoid easy-center-frame overrepresentation.
    g = frame[:, :, 1].astype("int32")
    col_energy = g.sum(axis=0)
    center_col = int(col_energy.argmax())
    edge = center_col < frame.shape[1] * 0.3 or center_col > frame.shape[1] * 0.7
    if not edge and (i // STRIDE) % 2 == 1:
        continue
    cv2.imwrite(str(OUT / f"f{i:06d}.jpg"), frame)
    saved += 1
    if saved >= MAX_FRAMES:
        break
cap.release()
print("saved", saved, "frames to", OUT)


In [ ]:
import shutil
shutil.make_archive("/content/pitch_frames", "zip", OUT)
from google.colab import files
files.download("/content/pitch_frames.zip")
